# Linear & Logistic Regression — Hands-On Notebook

This notebook is the practical companion to the slide deck. Every code cell has a comment on (almost) every line explaining exactly what that line does — so you can run it live with students, pause on any line, and explain it in plain English.

**We use one running example throughout:**
- `hours` = how many hours a student studied
- First we predict their exact exam `score` (**Linear Regression**)
- Then we flip it into a yes/no question: did they `pass`? (**Logistic Regression**)

Run each cell top to bottom with **Shift + Enter**. Feel free to change numbers and re-run cells to see what happens — that's the best way to build intuition.

## Setup — Import the Libraries We'll Use

In [ ]:
# pandas: lets us store and work with data in table form, like a spreadsheet
import pandas as pd

# numpy: gives us tools for numerical operations, like generating ranges of numbers
import numpy as np

# matplotlib: the library we'll use to draw charts and graphs
import matplotlib.pyplot as plt

# train_test_split: splits our data into a "training" part and a "testing" part
from sklearn.model_selection import train_test_split

# LinearRegression: the model we'll use in Part 1, to predict a NUMBER
from sklearn.linear_model import LinearRegression

# LogisticRegression: the model we'll use in Part 2, to predict a YES/NO category
from sklearn.linear_model import LogisticRegression

# these are the scoring tools we'll use to check how good our models are
from sklearn.metrics import r2_score, mean_squared_error, accuracy_score, confusion_matrix

print("All libraries loaded successfully!")  # if this prints with no errors above, we're ready to go

# Part 1 — Linear Regression

**Goal:** predict a student's exam `score` from how many `hours` they studied.

## Step 1 — Create the Data

In [ ]:
# We're typing the dataset in by hand so it's easy to follow along.
# In a real project this would usually be loaded from a CSV file instead.
data = pd.DataFrame({
    "hours": [1, 2, 3, 4, 5, 6, 7, 8, 9],           # how many hours each student studied
    "score": [35, 40, 50, 55, 65, 70, 78, 85, 92]   # the exam score that student got
})

data  # just typing a variable name on its own line displays it as a table in Jupyter/Colab

## Step 2 — Look at the Data First

In [ ]:
# Always look at your data before modeling it.
# A scatter plot shows each student as one dot: x = hours studied, y = exam score.
plt.scatter(data["hours"], data["score"], color="blue")

plt.xlabel("Hours Studied")      # label for the horizontal axis
plt.ylabel("Exam Score")         # label for the vertical axis
plt.title("Hours Studied vs. Exam Score")   # a title so the chart is self-explanatory

plt.show()   # actually renders the chart below this cell

Notice the dots roughly follow a straight line — as hours go up, score goes up too. That's exactly the kind of pattern Linear Regression is built to find.

## Step 3 — Prepare X (input) and y (target)

In [ ]:
# X must always be 2D (a table of feature columns) — that's why we use DOUBLE brackets,
# even though we only have one feature ("hours") right now.
X = data[["hours"]]

# y is 1D — it's the single column we are trying to predict.
y = data["score"]

print("Shape of X:", X.shape)   # (9, 1)  ->  9 rows, 1 feature column
print("Shape of y:", y.shape)   # (9,)    ->  9 target values

## Step 4 — Split into Training and Test Sets

In [ ]:
# We hold back 20% of the data as a "test set" that the model will NEVER see while learning.
# That way, later, we can honestly check how well it predicts data it has never encountered.
# random_state=42 just makes the split reproducible -- we'll get the same split every time we run this cell.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training rows:", len(X_train))   # how many students the model will learn from
print("Test rows:", len(X_test))        # how many students we'll use to check its predictions

## Step 5 — Create and Train the Model

In [ ]:
# Create a brand new Linear Regression model. Right now it knows nothing.
model = LinearRegression()

# .fit() is where the actual "learning" happens: the model works out the slope (w)
# and intercept (b) that make its predictions as close as possible to y_train.
model.fit(X_train, y_train)

# model.coef_ holds the slope(s) the model learned -- coef_[0] is the slope for "hours"
print("Slope (w):", model.coef_[0])

# model.intercept_ holds the intercept (b) -- the predicted score at 0 hours studied
print("Intercept (b):", model.intercept_)

## Step 6 — Make Predictions

In [ ]:
# Ask the trained model to predict scores for the test set -- data it has never seen before.
predictions = model.predict(X_test)

# Build a small table so we can compare the real scores to the model's guesses, side by side.
comparison = pd.DataFrame({
    "hours": X_test["hours"].values,     # the input hours from the test set
    "actual_score": y_test.values,       # what the student actually scored
    "predicted_score": predictions       # what the model predicted
})

comparison   # display the comparison table

## Step 7 — Evaluate the Model

In [ ]:
# R-squared (R2): what share of the pattern in the data our line explains.
# 1.0 = perfect fit. 0 = no better than just guessing the average score every time.
r2 = r2_score(y_test, predictions)

# Mean Squared Error (MSE): the average of (actual - predicted) squared, across all test points.
# Smaller is better -- it means predictions are closer to the real scores.
mse = mean_squared_error(y_test, predictions)

print("R-squared:", round(r2, 3))
print("Mean Squared Error:", round(mse, 2))

## Step 8 — Visualize the Fit Line

In [ ]:
# Plot every real data point as a blue dot (using the FULL dataset, not just the test set).
plt.scatter(X, y, color="blue", label="Actual data")

# Plot the model's predicted line across every hour value in our dataset, in red.
plt.plot(X, model.predict(X), color="red", label="Best-fit line")

plt.xlabel("Hours Studied")
plt.ylabel("Exam Score")
plt.title("Linear Regression: Best-Fit Line")
plt.legend()   # shows the little box explaining what blue dots and the red line mean

plt.show()

## Step 9 — Predict a Brand-New Value

In [ ]:
# Let's predict the score for a student who studied 6.5 hours -- a value NOT in our original data.
# It must be wrapped in a DataFrame with the SAME column name ("hours") the model was trained on.
new_hours = pd.DataFrame({"hours": [6.5]})

predicted_score = model.predict(new_hours)   # returns an array, e.g. [72.3]

print(f"Predicted score for 6.5 hours studied: {predicted_score[0]:.1f}")

### Bonus (optional) — Multiple Linear Regression

What if we had a second feature, like the number of practice tests a student took? Let's see how little the code changes.

In [ ]:
# Copy the original data so we don't accidentally modify it.
data2 = data.copy()

# Add a made-up second feature: how many practice tests each student took.
data2["practice_tests"] = [0, 1, 1, 2, 2, 3, 3, 4, 4]

# Now X has TWO columns instead of one -- everything else about the workflow stays identical.
X2 = data2[["hours", "practice_tests"]]
y2 = data2["score"]

model2 = LinearRegression()   # a brand new, separate model
model2.fit(X2, y2)            # train on both features at once -- no extra code needed!

print("Weights [hours, practice_tests]:", model2.coef_)
print("Intercept:", model2.intercept_)

# Part 2 — Logistic Regression

**Goal:** predict a yes/no answer -- did the student `pass`? -- from how many `hours` they studied.

## Step 1 — Create the Data

In [ ]:
# Same idea as before, but now the outcome is a category, not a number:
# 0 = failed, 1 = passed.
data_c = pd.DataFrame({
    "hours": [1, 2, 3, 4, 5, 6, 7, 8, 9],
    "passed": [0, 0, 0, 0, 1, 1, 1, 1, 1]   # 0 = fail, 1 = pass
})

data_c   # display the table

## Step 2 — Look at the Data First

In [ ]:
# Plot each student: x = hours studied, y = passed (1) or failed (0).
plt.scatter(data_c["hours"], data_c["passed"], color="green")

plt.xlabel("Hours Studied")
plt.ylabel("Passed (1) or Failed (0)")
plt.title("Hours Studied vs. Pass/Fail")

plt.yticks([0, 1])   # force the y-axis to only show 0 and 1, since those are the only real values

plt.show()

There's no straight line that sensibly fits 0/1 data like this -- a line would predict values below 0 or above 1, which make no sense as a yes/no answer. That's exactly why Logistic Regression exists: it fits an S-shaped curve instead, which we'll see in Step 8.

## Step 3 — Prepare X and y

In [ ]:
X_c = data_c[["hours"]]   # features: hours studied (2D, same rule as before)
y_c = data_c["passed"]    # target: 0 or 1

print("Shape of X_c:", X_c.shape)
print("Shape of y_c:", y_c.shape)

## Step 4 — Split into Training and Test Sets

In [ ]:
# Same reasoning as Part 1: hold back 20% of the data to test on honestly, later.
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_c, y_c, test_size=0.2, random_state=42
)

print("Training rows:", len(X_train_c))
print("Test rows:", len(X_test_c))

## Step 5 — Create and Train the Model

In [ ]:
# Create a brand new Logistic Regression model.
model_c = LogisticRegression()

# .fit() trains the model -- internally it uses Log Loss (not MSE) to judge itself,
# since we're now predicting a probability/category, not a plain number.
model_c.fit(X_train_c, y_train_c)

print("Model trained!")

## Step 6 — Make Predictions: Classes and Probabilities

In [ ]:
# .predict() gives the FINAL yes/no answer (0 or 1), after applying the 0.5 threshold internally.
class_predictions = model_c.predict(X_test_c)

# .predict_proba() gives the RAW probabilities behind that answer.
# It returns two columns per row: [probability of class 0, probability of class 1].
probabilities = model_c.predict_proba(X_test_c)

comparison_c = pd.DataFrame({
    "hours": X_test_c["hours"].values,
    "actual": y_test_c.values,
    "predicted_class": class_predictions,
    "probability_of_pass": probabilities[:, 1]   # keep just the "probability of passing" column
})

comparison_c   # display the comparison table

## Step 7 — Evaluate: Accuracy & Confusion Matrix

In [ ]:
# Accuracy: the percentage of test predictions that were exactly correct.
accuracy = accuracy_score(y_test_c, class_predictions)
print("Accuracy:", accuracy)

# Confusion matrix: a 2x2 table showing exactly WHICH mistakes were made, not just how many.
# By scikit-learn's convention: rows = actual values, columns = predicted values.
cm = confusion_matrix(y_test_c, class_predictions)
print("Confusion Matrix (raw):\n", cm)

In [ ]:
# Wrap the confusion matrix in a labeled table so it's much easier to read and explain.
cm_labeled = pd.DataFrame(
    cm,
    index=["Actual: Fail", "Actual: Pass"],        # row labels
    columns=["Predicted: Fail", "Predicted: Pass"]  # column labels
)

cm_labeled   # display the labeled confusion matrix

## Step 8 — Visualize the Sigmoid Curve

In [ ]:
# Generate 200 evenly-spaced hour values from 0 to 10, purely so we can draw a SMOOTH curve.
hours_range = np.linspace(0, 10, 200).reshape(-1, 1)   # reshape makes it 2D, as sklearn expects

# Ask the model for the probability of passing at every one of those 200 hour values.
probs_range = model_c.predict_proba(hours_range)[:, 1]

# Plot the S-shaped probability curve.
plt.plot(hours_range, probs_range, color="teal", label="Predicted probability of passing")

# Plot the real pass/fail data points on top, so we can see how the curve fits them.
plt.scatter(data_c["hours"], data_c["passed"], color="blue", label="Actual data")

# Draw a dashed horizontal line at the 0.5 decision threshold, for reference.
plt.axhline(0.5, color="orange", linestyle="--", label="0.5 threshold")

plt.xlabel("Hours Studied")
plt.ylabel("Probability of Passing")
plt.title("Logistic Regression: The Sigmoid Curve")
plt.legend()

plt.show()

## Step 9 — Predict a Brand-New Value

In [ ]:
# Let's check a student who studied 4.5 hours -- right around our decision boundary.
new_student = pd.DataFrame({"hours": [4.5]})

predicted_class = model_c.predict(new_student)         # the final 0/1 answer
predicted_prob = model_c.predict_proba(new_student)     # the underlying probabilities

print("Predicted class (0 = fail, 1 = pass):", predicted_class[0])
print(f"Probability of passing: {predicted_prob[0][1]:.2f}")

## Recap

**Linear Regression**
- Predicts a NUMBER by fitting a straight line: `y = wx + b`
- Trained by minimizing Mean Squared Error
- Evaluated with R-squared and MSE

**Logistic Regression**
- Predicts a YES/NO category by fitting an S-shaped (sigmoid) curve
- Trained by minimizing Log Loss
- Evaluated with Accuracy and the Confusion Matrix

**The workflow is nearly identical for both:**
load data → look at it → prepare X and y → split → `.fit()` → `.predict()` → evaluate → visualize.

---

### Try It Yourself (optional)

Add a new code cell below and try:
1. Change the `score` or `passed` values in the data and re-run everything -- how do the slope, R-squared, or accuracy change?
2. Predict the score / pass probability for a student who studied 10 hours, or 0.5 hours.
3. In Part 1's bonus section, add a third feature and see how the model handles it.